# 📓 Notebook 13 — SQL Fundamentals with pandas

> **Module:** Data Engineering · **Estimated time:** 50–70 min · **Difficulty:** Beginner / Intermediate

Every analyst job description asks for **SQL**. Every modern data product uses it. And yet you can read 90% of the Python data-science internet without seeing a single `SELECT` statement.

This notebook closes that gap. We will load the AI-support-bot data into a tiny SQLite database (which ships with Python — no install), write SQL queries against it, do the same analyses in pandas, and learn when to reach for which. By the end you will be reading and writing SQL fluently against a database that lives entirely inside this notebook.

## 🎯 Learning objectives

By the end of this notebook you can:

1. Connect to a **SQLite** database from Python and create tables from a DataFrame.
2. Write the **six SQL clauses** that cover ~95% of analytics work: `SELECT`, `FROM`, `WHERE`, `GROUP BY`, `ORDER BY`, `LIMIT`.
3. **Aggregate** with `COUNT`, `SUM`, `AVG`, `MIN`, `MAX`.
4. **Join** two tables on a key.
5. Use **CTEs** (`WITH ...`) to break a complex query into readable steps.
6. Move data between SQL and pandas with `read_sql` and `to_sql`.
7. Decide when SQL is the better tool than pandas (and vice versa).

## ✅ Prerequisites

Notebooks 1–5 (especially NB5 — DataFrames, group-by). No prior SQL needed.

> 💡 **Why SQLite?** It is a serverless SQL database built into Python's standard library. No installation, no server. The exact same SQL you write here works (with a few syntactic differences) in PostgreSQL, MySQL, BigQuery, Snowflake, and every other SQL system you'll meet professionally.

## 1. SQL in one slide

SQL describes data manipulation as a sequence of **clauses** that operate on tables. The most common shape:

```
SELECT   columns_or_aggregations         ← what you want
FROM     table                            ← where to look
WHERE    row_filter                       ← which rows
GROUP BY column(s)                        ← bucket rows together
HAVING   aggregate_filter                 ← which buckets
ORDER BY column(s)                        ← sort the result
LIMIT    n                                ← top-n only
```

You won't use every clause every time. A simple "show me the first ten rows" is just `SELECT * FROM table LIMIT 10`.

The mental model: **SQL describes *what* you want, the database figures out *how*.** That declarative style is the source of both its power and its quirks.

## 2. Setup — load a CSV into SQLite

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

# Build a fresh synthetic dataset inline (so the notebook is self-contained)
import numpy as np
RNG = np.random.default_rng(42)

channels = ["Email", "Chat", "Phone", "Web Form", "Social"]
months   = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
profile = {
    "Email":    (0.55, 0.018, 4_200,  3_500, 4.10),
    "Chat":     (0.72, 0.012, 6_500,    900, 4.35),
    "Phone":    (0.18, 0.008, 2_100, 14_000, 3.85),
    "Web Form": (0.65, 0.015, 3_300,  6_000, 4.00),
    "Social":   (0.35, 0.030, 1_400,  2_400, 3.95),
}
rows = []
for ch in channels:
    a0, g, vol, base_lat, base_sat = profile[ch]
    for i, m in enumerate(months):
        a = max(0.05, min(0.95, a0 + i * g + RNG.normal(0, 0.015)))
        v = int(vol * (1 + 0.01 * i + RNG.normal(0, 0.03)))
        lat = base_lat * (1 - 0.012 * i) * (1 + RNG.normal(0, 0.05))
        sat = max(1.0, min(5.0, base_sat + 0.4 * (a - a0)
              - 0.000_004 * (lat - base_lat) + RNG.normal(0, 0.05)))
        cost = max(0.15, a * 0.30 + (1 - a) * 5.50 + RNG.normal(0, 0.10))
        rows.append({"channel": ch, "month": m, "month_num": i+1,
                     "tickets_total": v, "tickets_auto": int(v*a),
                     "automation_rate": round(a, 3),
                     "latency_ms": int(lat), "satisfaction": round(sat, 2),
                     "cost_per_ticket": round(cost, 2)})
support_ops = pd.DataFrame(rows)

# Create an in-memory SQLite database (no file on disk — instant)
conn = sqlite3.connect(":memory:")

# Push the DataFrame into a table
support_ops.to_sql("support_ops", conn, index=False, if_exists="replace")

print(f"✅ Loaded {len(support_ops)} rows into table 'support_ops'")
print(f"   Columns: {list(support_ops.columns)}")


**What just happened.**

- `sqlite3.connect(":memory:")` opens a database that exists only in RAM — perfect for teaching.
- `df.to_sql("name", conn)` is the pandas → SQL bridge. Reverses with `pd.read_sql(...)`, shown next.

> 💡 **In real life** you'd use `sqlite3.connect("ops.db")` to persist the database to a file, or `sqlalchemy.create_engine("postgresql://...")` for a real production database. The rest of this notebook is identical regardless.

## 3. Your first SELECT

The `pd.read_sql(query, conn)` function runs a SQL query and gives you the result as a DataFrame. Let's use it to peek at the table.

In [ ]:
# The simplest possible query — "give me the first 5 rows"
q = "SELECT * FROM support_ops LIMIT 5"
pd.read_sql(q, conn)


In [ ]:
# Just the columns we care about
q = '''
SELECT channel, month, automation_rate, satisfaction
FROM support_ops
LIMIT 5
'''
pd.read_sql(q, conn)


**Conventions worth picking up immediately:**

- **Uppercase keywords** (`SELECT`, `FROM`, `WHERE`) and **lowercase identifiers** (`channel`, `support_ops`). The DB doesn't care, but humans read SQL faster when it follows this convention.
- **One clause per line** for anything longer than a one-liner. Vertical SQL is much easier to debug.

## 4. WHERE — filtering rows

`WHERE` is SQL's row filter — same job as a pandas boolean mask.

In [ ]:
# All months where the bot was working very well (high automation, high satisfaction)
q = '''
SELECT channel, month, automation_rate, satisfaction
FROM   support_ops
WHERE  automation_rate >= 0.80
   AND satisfaction    >= 4.30
ORDER BY automation_rate DESC
'''
pd.read_sql(q, conn)


In [ ]:
# String comparisons — note the single quotes around the literal
q = '''
SELECT channel, month, latency_ms
FROM   support_ops
WHERE  channel = 'Phone'
   AND latency_ms > 10000
ORDER BY latency_ms DESC
LIMIT 5
'''
pd.read_sql(q, conn)


**SQL operators that show up daily:**

| Operator | Meaning | Example |
|---|---|---|
| `=`, `<>`, `<`, `>`, `<=`, `>=` | comparisons | `WHERE channel = 'Chat'` |
| `AND`, `OR`, `NOT` | combine conditions | `WHERE a > 5 AND b < 10` |
| `IN (...)` | match any of a list | `WHERE channel IN ('Email','Chat')` |
| `BETWEEN x AND y` | range | `WHERE month_num BETWEEN 4 AND 6` |
| `LIKE 'pat%'` | string pattern (`%` = wildcard) | `WHERE channel LIKE 'E%'` |
| `IS NULL` / `IS NOT NULL` | missing values | `WHERE satisfaction IS NOT NULL` |

> ⚠️ **Use `IS NULL`, not `= NULL`.** SQL's three-valued logic (`TRUE` / `FALSE` / `UNKNOWN`) treats `column = NULL` as always *unknown*, so it never matches anything.

## 5. Aggregations — the part that earns its keep

The five core aggregation functions — `COUNT`, `SUM`, `AVG`, `MIN`, `MAX` — combined with `GROUP BY` produce most of the analytical output a business consumes.

In [ ]:
# Total tickets and average automation rate, per channel
q = '''
SELECT channel,
       COUNT(*)               AS n_months,
       SUM(tickets_total)     AS total_tickets,
       AVG(automation_rate)   AS mean_auto_rate,
       AVG(satisfaction)      AS mean_satisfaction
FROM     support_ops
GROUP BY channel
ORDER BY total_tickets DESC
'''
pd.read_sql(q, conn).round(3)


**Reading the query above:**

- `GROUP BY channel` buckets the 60 rows into 5 groups (one per channel).
- Each `AGG(...)` is computed *per bucket*.
- `AS column_name` renames the output — much clearer than the default `AVG(automation_rate)`.

> 🎯 **The rule of `GROUP BY`.** Every column in the `SELECT` list must either be in the `GROUP BY` clause **or** be wrapped in an aggregation function. Mixing the two without grouping is a beginner bug the database will (eventually) refuse to run.

### `HAVING` — filtering on aggregates

`WHERE` filters *rows*. `HAVING` filters *groups*. You usually want one or the other, sometimes both.

In [ ]:
# Channels with average automation rate > 60%
q = '''
SELECT channel,
       AVG(automation_rate) AS mean_auto
FROM     support_ops
GROUP BY channel
HAVING   AVG(automation_rate) > 0.60
ORDER BY mean_auto DESC
'''
pd.read_sql(q, conn).round(3)


## 6. Grouping by multiple columns and time

`GROUP BY` can take several columns — useful when you want a cross-tab.

In [ ]:
# Quarterly automation rate per channel
q = '''
SELECT channel,
       CASE
         WHEN month_num <= 3 THEN 'Q1'
         WHEN month_num <= 6 THEN 'Q2'
         WHEN month_num <= 9 THEN 'Q3'
         ELSE 'Q4'
       END                          AS quarter,
       AVG(automation_rate)         AS mean_auto
FROM     support_ops
GROUP BY channel, quarter
ORDER BY channel, quarter
'''
pd.read_sql(q, conn).round(3)


> 💡 **`CASE WHEN ... THEN ... ELSE ... END`** is SQL's if-elif-else. It runs once per row and returns a value. We just used it to turn a continuous month number into a categorical quarter — feature engineering in pure SQL.

## 7. JOINs — bringing two tables together

Real databases have many tables. Imagine we have a separate table of **channel metadata** — the team manager, the channel's launch year, and so on. Let's create one and join it back.

In [ ]:
# A small "dimension" table about each channel
channel_meta = pd.DataFrame([
    {"channel": "Email",    "team_lead": "Anna",  "launched_year": 2018, "is_voice": 0},
    {"channel": "Chat",     "team_lead": "Bilal", "launched_year": 2020, "is_voice": 0},
    {"channel": "Phone",    "team_lead": "Carla", "launched_year": 2010, "is_voice": 1},
    {"channel": "Web Form", "team_lead": "Diego", "launched_year": 2019, "is_voice": 0},
    {"channel": "Social",   "team_lead": "Elena", "launched_year": 2023, "is_voice": 0},
])
channel_meta.to_sql("channel_meta", conn, index=False, if_exists="replace")

# Now join: each row of support_ops gets the team_lead and launched_year attached
q = '''
SELECT s.channel,
       s.month,
       s.automation_rate,
       m.team_lead,
       m.launched_year
FROM   support_ops AS s
JOIN   channel_meta AS m ON s.channel = m.channel
WHERE  s.month_num = 12         -- December only
ORDER BY s.automation_rate DESC
'''
pd.read_sql(q, conn).round(3)


**The four kinds of JOIN, in plain English:**

| Type | Keeps … |
|---|---|
| `INNER JOIN` (default) | rows that match in **both** tables |
| `LEFT JOIN`            | every row from the left, with `NULL` where the right has no match |
| `RIGHT JOIN`           | the mirror image (rare in modern SQL — flip your tables) |
| `FULL OUTER JOIN`      | every row from either side; matches where possible |

> 🎯 **99% of the time you want `INNER JOIN` or `LEFT JOIN`.** Reach for `LEFT JOIN` whenever the right table might be missing rows for some keys and you don't want to drop those rows.

In [ ]:
# A LEFT JOIN — useful when the right table might be missing entries
extra = pd.DataFrame([{"channel": "Email", "monthly_budget_usd": 12000}])
extra.to_sql("budgets", conn, index=False, if_exists="replace")

q = '''
SELECT s.channel,
       AVG(s.cost_per_ticket) AS mean_cost,
       b.monthly_budget_usd
FROM     support_ops AS s
LEFT JOIN budgets AS b ON s.channel = b.channel
GROUP BY s.channel
'''
pd.read_sql(q, conn).round(3)


See how channels without a budget show `None` instead of being dropped. That's the `LEFT JOIN` paying its way — you preserve all channels even when the budget table only knows about one.

## 8. CTEs — making complex queries readable

A **CTE** (Common Table Expression) lets you give a name to an intermediate query and reuse it. It is the SQL equivalent of breaking a long Python function into helper functions.

In [ ]:
# Without a CTE — works, but hard to read
q_inline = '''
SELECT channel, mean_auto, mean_sat
FROM (
  SELECT channel,
         AVG(automation_rate) AS mean_auto,
         AVG(satisfaction)    AS mean_sat
  FROM   support_ops
  GROUP BY channel
)
WHERE mean_auto > 0.5 AND mean_sat > 4.0
'''
print("Inline subquery result:")
print(pd.read_sql(q_inline, conn).round(3))

# Same query with a CTE — much easier to read and to extend
q_cte = '''
WITH channel_summary AS (
    SELECT channel,
           AVG(automation_rate) AS mean_auto,
           AVG(satisfaction)    AS mean_sat
    FROM   support_ops
    GROUP BY channel
)
SELECT *
FROM   channel_summary
WHERE  mean_auto > 0.5 AND mean_sat > 4.0
ORDER BY mean_auto DESC
'''
print("\nCTE result:")
print(pd.read_sql(q_cte, conn).round(3))


> 💡 **When in doubt, CTE.** A 5-line CTE that you read top-to-bottom always beats a 5-level-deep nested subquery. Modern SQL style guides recommend CTEs for anything beyond a single `SELECT`.

## 9. SQL vs pandas — when to use which

You can do almost any analysis in either tool. So when do you reach for which?

| Use **SQL** when … | Use **pandas** when … |
|---|---|
| The data lives in a database — *don't move it before you have to*. | The data is already a DataFrame. |
| Joining 2–10 large tables on keys. | Reshaping a single dataset (pivot, melt, multi-index). |
| Filtering / aggregating "most" of a large table before bringing it into Python. | Doing per-row, per-column transformations that are awkward in SQL. |
| Sharing the query with non-Python colleagues. | Plugging into matplotlib, scikit-learn, etc. |
| The result needs to be a stable view that anyone can re-run. | Quick exploration, prototyping. |

> 🎯 **A common pro pattern:** *use SQL for the heavy lift, pandas for the last mile.* Pull a small filtered/aggregated table out of the database with SQL, then do the plotting / modelling in pandas. You get the best of both.

In [ ]:
# Same analysis, two ways — pick the one that reads better
# A) Pure SQL
print("--- SQL ---")
print(pd.read_sql('''
    SELECT channel,
           AVG(automation_rate) AS mean_auto
    FROM   support_ops
    GROUP BY channel
    ORDER BY mean_auto DESC
''', conn).round(3))

# B) Pure pandas
print("\n--- pandas ---")
print(support_ops.groupby("channel")["automation_rate"]
                 .mean().sort_values(ascending=False)
                 .round(3).to_frame())


For this tiny query they look equally readable. For a 4-table join with three CTEs and a window function — SQL wins on clarity. For a chained `.assign(...).pivot(...).rolling(...).plot()` — pandas wins.

## 10. Tiny tour of window functions

Window functions compute an aggregate **per row** using a "window" of nearby rows — running totals, ranks, lag/lead. They are SQL's secret weapon.

In [ ]:
# Rank each channel's months from best to worst by automation rate
q = '''
SELECT channel,
       month,
       automation_rate,
       RANK() OVER (PARTITION BY channel ORDER BY automation_rate DESC) AS auto_rank
FROM support_ops
ORDER BY channel, auto_rank
'''
pd.read_sql(q, conn).head(15)


The output has three months per channel where `auto_rank = 1` would be the best month for that channel, etc. That's a *ranking inside each group* — a pattern that costs you a `groupby + transform` dance in pandas but one line of SQL.

Window-function shapes worth knowing:

- `RANK() OVER (...)` — competition ranking (ties share a rank).
- `ROW_NUMBER() OVER (...)` — unique 1, 2, 3, … even for ties.
- `SUM(x) OVER (PARTITION BY g ORDER BY t)` — running total within each group.
- `LAG(x, 1) OVER (ORDER BY t)` — value from the previous row.

## 11. Cleaning up

In [ ]:
conn.close()
print("Connection closed.")


## 🧪 Practice exercises

### Exercise 1 — Recreate the connection and run a query

Reopen the connection, recreate the `support_ops` table from the DataFrame, then write a query that returns the **mean cost-per-ticket per channel**, sorted ascending.

In [ ]:
# Your code here  👇
conn = sqlite3.connect(":memory:")
support_ops.to_sql("support_ops", conn, index=False, if_exists="replace")


<details>
<summary>💡 <b>Solution</b></summary>

```python
q = '''
SELECT channel,
       AVG(cost_per_ticket) AS mean_cost
FROM     support_ops
GROUP BY channel
ORDER BY mean_cost ASC
'''
pd.read_sql(q, conn).round(2)
```

Chat should come out cheapest, Phone most expensive. The pattern is **`SELECT … FROM … GROUP BY … ORDER BY`** — the four clauses you'll use in every other analytical query you write.
</details>

### Exercise 2 — A two-condition filter

Write a query that returns every (channel, month) where **automation_rate > 0.75 AND latency_ms < 2000**. Sort by automation_rate descending.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
q = '''
SELECT channel, month, automation_rate, latency_ms
FROM   support_ops
WHERE  automation_rate > 0.75
   AND latency_ms      < 2000
ORDER BY automation_rate DESC
'''
pd.read_sql(q, conn).round(3)
```

These are the "fast + accurate" months — your *high-leverage* combinations of conditions.
</details>

### Exercise 3 — Best month per channel (window function)

Use a window function to find each channel's **single best month** (highest satisfaction). The result should have exactly 5 rows.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
q = '''
WITH ranked AS (
    SELECT channel, month, satisfaction,
           ROW_NUMBER() OVER (PARTITION BY channel
                              ORDER BY satisfaction DESC) AS rn
    FROM support_ops
)
SELECT channel, month, satisfaction
FROM ranked
WHERE rn = 1
ORDER BY satisfaction DESC
'''
pd.read_sql(q, conn).round(2)
```

**Pattern.** Rank rows inside each group, then keep only `rn = 1`. This "top-N per group" pattern is a window-function classic — and the reason every senior analyst learns them.
</details>

### Exercise 4 — Join with the metadata table

Recreate `channel_meta` (as we did in §7), then write a query that returns the **mean automation rate** per channel together with its **team lead** and **launched year**, sorted by automation rate descending.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
channel_meta = pd.DataFrame([
    {"channel": "Email",    "team_lead": "Anna",  "launched_year": 2018},
    {"channel": "Chat",     "team_lead": "Bilal", "launched_year": 2020},
    {"channel": "Phone",    "team_lead": "Carla", "launched_year": 2010},
    {"channel": "Web Form", "team_lead": "Diego", "launched_year": 2019},
    {"channel": "Social",   "team_lead": "Elena", "launched_year": 2023},
])
channel_meta.to_sql("channel_meta", conn, index=False, if_exists="replace")

q = '''
SELECT s.channel,
       AVG(s.automation_rate) AS mean_auto,
       m.team_lead,
       m.launched_year
FROM     support_ops AS s
JOIN     channel_meta AS m ON s.channel = m.channel
GROUP BY s.channel, m.team_lead, m.launched_year
ORDER BY mean_auto DESC
'''
pd.read_sql(q, conn).round(3)
```

**Pattern used.** Join → group → aggregate → sort. This is the shape of every "X by Y, enriched with Z" report your stakeholders will ever ask for.
</details>

### Exercise 5 — Debug me 🐞

The query below is supposed to return channels with **more than 50K total tickets**, but it raises an error. Find the bug.

```sql
SELECT channel
FROM   support_ops
WHERE  SUM(tickets_total) > 50000
GROUP BY channel
```

In [ ]:
# Your fixed query  👇


<details>
<summary>💡 <b>Solution</b></summary>

`WHERE` filters individual rows **before** aggregation, so it cannot reference an aggregate like `SUM(...)`. Filtering on aggregates is the job of `HAVING`, which runs **after** the group is formed.

```python
q = '''
SELECT channel,
       SUM(tickets_total) AS total
FROM   support_ops
GROUP BY channel
HAVING SUM(tickets_total) > 50000
ORDER BY total DESC
'''
pd.read_sql(q, conn)
```

**Remember the order of operations in SQL:** `FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT`. Knowing this order makes 90% of confusing errors instantly obvious.
</details>

## 🧠 Stretch exercises

Two more applied exercises to deepen the material. Try them yourself before opening the solution.


### Stretch exercise A — Running cumulative totals

Using a window function, return — for each (channel, month_num) — the **running cumulative ticket count** within that channel. Sort by channel, then month_num.


<details>
<summary>💡 <b>Solution</b></summary>

```python
q = '''
SELECT channel,
       month_num,
       tickets_total,
       SUM(tickets_total) OVER (PARTITION BY channel ORDER BY month_num) AS cumulative_tickets
FROM   support_ops
ORDER BY channel, month_num
'''
pd.read_sql(q, conn).head(15)
```

**Pattern used.** `SUM(...) OVER (PARTITION BY g ORDER BY t)` is the
canonical running-total expression. It's the SQL analogue of
`df.groupby("channel")["tickets"].cumsum()` — *and* it's how every
financial report computes year-to-date numbers.

</details>

### Stretch exercise B — Self-join to compute month-over-month deltas

Use a self-join (or `LAG()`) to return each (channel, month_num) with its **month-over-month change** in automation_rate. The first month of each channel should have `delta = NULL`.


<details>
<summary>💡 <b>Solution</b></summary>

```python
q = '''
SELECT channel,
       month_num,
       automation_rate,
       automation_rate - LAG(automation_rate) OVER (
           PARTITION BY channel ORDER BY month_num
       ) AS delta
FROM support_ops
ORDER BY channel, month_num
'''
pd.read_sql(q, conn).head(15)
```

**`LAG()` is one of the most useful window functions** — it gives
you the previous row's value within a partition. Same shape as
`series.diff(1)` in pandas. The first row per channel is `NULL`
because there's no prior month.

</details>

## 🎁 Bonus mini-project — A SQL-driven mini report

Using only SQL (then convert the final result to a DataFrame), build a report with these columns: `channel`, `team_lead`, `total_tickets`, `mean_auto`, `mean_cost`. Sort by `total_tickets` descending. Use a `CTE` for the aggregation and a `JOIN` for the metadata.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
q = '''
WITH agg AS (
    SELECT channel,
           SUM(tickets_total)   AS total_tickets,
           AVG(automation_rate) AS mean_auto,
           AVG(cost_per_ticket) AS mean_cost
    FROM   support_ops
    GROUP BY channel
)
SELECT a.channel,
       m.team_lead,
       a.total_tickets,
       a.mean_auto,
       a.mean_cost
FROM     agg AS a
JOIN     channel_meta AS m ON a.channel = m.channel
ORDER BY a.total_tickets DESC
'''
pd.read_sql(q, conn).round(3)
```

**What you built.** The same kind of one-page report a manager asks for at the end of every quarter — and the SQL is *legible*. A junior analyst could read it without asking what each step does. That readability is the real product of CTEs.
</details>

## 🧠 Key takeaways

1. **SQL is declarative**: you describe *what* you want, the database figures out *how*.
2. The six core clauses — `SELECT`, `FROM`, `WHERE`, `GROUP BY`, `ORDER BY`, `LIMIT` — cover most analytics work.
3. **`HAVING` filters aggregates; `WHERE` filters rows.** Use them in the right place.
4. **Every column in `SELECT` must be in `GROUP BY` or wrapped in an aggregation.**
5. **`INNER JOIN`** for "must match"; **`LEFT JOIN`** for "may be missing".
6. **CTEs (`WITH ... AS`)** make complex queries readable — use them freely.
7. **Window functions** let you compute aggregates per row — the secret weapon for top-N-per-group and running totals.
8. **SQL vs pandas:** SQL for joins and heavy aggregation; pandas for reshape, plot, model. They're complements, not rivals.

## ✅ Self-assessment

- [ ] Load a DataFrame into a SQLite table and read it back as a DataFrame
- [ ] Write a `SELECT ... WHERE ... ORDER BY ... LIMIT` query
- [ ] Aggregate with `GROUP BY` and the five basic aggregation functions
- [ ] Filter aggregates with `HAVING`
- [ ] Inner-join and left-join two tables
- [ ] Use a CTE (`WITH ...`) to structure a multi-step query
- [ ] Explain when SQL beats pandas and vice versa

## 🚀 Next step

Continue with **Notebook 14 — Time Series and Forecasting Basics**, where the monthly data you've been querying becomes the input to a real forecast: "what's the automation rate likely to be three months from now?".